# Task B — v1 sequence model (ESM-2) and abundance+sequence fusion

**Question this notebook answers:** does protein *sequence* carry bioactivity signal *beyond* what the abundance profile already provides? This is the clean test that the v0 abundance model could not give (because of the abundance/label circularity documented in `RESULTS.md` §4.4).

**Three models, all on the identical family split as `02_abundance_only.ipynb`:**
1. **Sequence-only** ranker: protein embedding → MLP → score.
2. **Abundance-only** (v0): reloaded from `predictions/02_mlp.pt` — the strong baseline to add to.
3. **Fusion**: a small calibrator trained on the validation split that combines the abundance score and the sequence score; evaluated on the test split.

**Embedding backend is swappable (`EMBED_BACKEND`):**
- `'kmer'` — amino-acid k-mer composition (pure numpy, no downloads). Runs anywhere, used for the **local smoke test** and as a legitimate classical sequence baseline.
- `'esm'` — real ESM-2 embeddings. Heavy: meant for the **CSUC cluster**. For 1.45 M sequences, precompute embeddings once with the standalone script `esm_embed.py` (see last section) and set `EMBED_BACKEND='precomputed'` to load `processed/seq_emb.npy`.
- `'precomputed'` — load `processed/seq_emb.npy` produced on the cluster.

**Local smoke test:** set `SMOKE=True` → subsamples families, uses the k-mer backend, verifies the full pipeline end-to-end in ~1 min on a laptop.

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score, roc_auc_score

torch.manual_seed(0); np.random.seed(0)

# ---- config ----
SMOKE = False                 # True: subsample + kmer backend, verify pipeline fast on laptop
EMBED_BACKEND = 'precomputed'  # 'kmer' | 'esm' | 'precomputed'
KMER_K = 2                   # 2 -> 400-dim, 3 -> 8000-dim composition vector
SMOKE_N = 60_000             # families to keep in SMOKE mode
ESM_MODEL = 'facebook/esm2_t12_35M_UR50D'  # used only if EMBED_BACKEND=='esm' (small; cluster uses 650M)
SEQ_EMB_FILE = 'seq_emb_35M.npy'  # precomputed ESM-2 embeddings (CSUC H100); 35M->480d, 650M->1280d

PROC = Path('processed')
PRED = Path('predictions'); PRED.mkdir(exist_ok=True)
device = ('mps' if torch.backends.mps.is_available()
          else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'device={device}  SMOKE={SMOKE}  EMBED_BACKEND={EMBED_BACKEND}')

device=mps  SMOKE=False  EMBED_BACKEND=precomputed


In [2]:
labels = pd.read_parquet(PROC / 'labels.parquet')
fids = pd.read_parquet(PROC / 'family_ids.parquet')['family_id'].values
seqs = pd.read_parquet(PROC / 'sequences.parquet')
assert (labels['family_id'].values == fids).all(), 'label/abundance family order mismatch'

# align sequences to the family order of labels/abundance
seq_map = dict(zip(seqs['family_id'], seqs['sequence']))
seq_arr = np.array([seq_map.get(f, '') for f in fids], dtype=object)
y_all = labels['is_bioactive'].astype(np.int8).values
print(f'families: {len(fids):,}  positives: {int(y_all.sum()):,}  base rate: {y_all.mean():.4f}')
print(f'sequences present: {(seq_arr != "").sum():,}')

families: 1,447,952  positives: 27,034  base rate: 0.0187
sequences present: 1,447,952


## Reproduce the *identical* split used in `02_abundance_only.ipynb`

Same `train_test_split` calls, same `random_state=0`, so the test families match exactly — this is what makes the sequence vs abundance comparison valid. In SMOKE mode we then subsample within each split (stratified) for speed.

In [3]:
from sklearn.model_selection import train_test_split
idx = np.arange(len(y_all))
tr, te = train_test_split(idx, test_size=0.15, stratify=y_all, random_state=0)
tr, va = train_test_split(tr,  test_size=0.15 / 0.85, stratify=y_all[tr], random_state=0)

# sanity: test families match the saved v0 predictions
v0 = pd.read_parquet(PRED / '02_abundance_only.parquet') if (PRED / '02_abundance_only.parquet').exists() else None
if v0 is not None:
    same = set(fids[te]) == set(v0['family_id'])
    print(f'test split matches 02 predictions: {same}')

def subsample(ix, n):
    if not SMOKE or len(ix) <= n: return ix
    pos = ix[y_all[ix] == 1]; neg = ix[y_all[ix] == 0]
    keep_neg = np.random.default_rng(0).choice(neg, size=max(n - len(pos), 1), replace=False)
    out = np.concatenate([pos, keep_neg]); np.random.default_rng(0).shuffle(out); return out

if SMOKE:
    frac = SMOKE_N / len(idx)
    tr = subsample(tr, int(len(tr) * frac * 7))
    va = subsample(va, int(len(va) * frac * 7))
    te = subsample(te, int(len(te) * frac * 7))
for nm, ix in [('train', tr), ('val', va), ('test', te)]:
    print(f'  {nm:5s}: {len(ix):>8,}  pos={int(y_all[ix].sum()):>6,}  rate={y_all[ix].mean():.4f}')

test split matches 02 predictions: True
  train: 1,013,566  pos=18,924  rate=0.0187
  val  :  217,193  pos= 4,055  rate=0.0187
  test :  217,193  pos= 4,055  rate=0.0187


## Sequence embeddings

`embed_sequences(seq_list)` returns an `(n, d)` float32 matrix. The k-mer backend is a normalized amino-acid k-mer frequency vector; the ESM backend mean-pools the last hidden state of ESM-2 (batched, no grad). On the cluster you would normally precompute ESM embeddings once with `esm_embed.py` and use `EMBED_BACKEND='precomputed'`.

In [4]:
AA = 'ACDEFGHIKLMNPQRSTVWY'
AA_IDX = {a: i for i, a in enumerate(AA)}

def kmer_embed(seq_list, k=KMER_K):
    from itertools import product
    kmers = [''.join(p) for p in product(AA, repeat=k)]
    kmer_idx = {km: i for i, km in enumerate(kmers)}
    out = np.zeros((len(seq_list), len(kmers)), dtype=np.float32)
    for i, s in enumerate(seq_list):
        s = ''.join(ch for ch in s if ch in AA_IDX)
        if len(s) < k: continue
        for j in range(len(s) - k + 1):
            km = s[j:j+k]
            idx_ = kmer_idx.get(km)
            if idx_ is not None: out[i, idx_] += 1
        tot = out[i].sum()
        if tot > 0: out[i] /= tot
    return out

def esm_embed(seq_list, model_name=ESM_MODEL, batch=8, max_len=1022):
    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()
    dim = model.config.hidden_size
    out = np.zeros((len(seq_list), dim), dtype=np.float32)
    with torch.no_grad():
        for s in range(0, len(seq_list), batch):
            chunk = [seq[:max_len] for seq in seq_list[s:s+batch]]
            enc = tok(chunk, return_tensors='pt', padding=True, truncation=True, max_length=max_len)
            enc = {kk: vv.to(device) for kk, vv in enc.items()}
            hs = model(**enc).last_hidden_state           # (b, L, d)
            mask = enc['attention_mask'].unsqueeze(-1)      # (b, L, 1)
            pooled = (hs * mask).sum(1) / mask.sum(1).clamp(min=1)
            out[s:s+len(chunk)] = pooled.cpu().numpy()
            if s % (batch * 50) == 0: print(f'    esm {s:,}/{len(seq_list):,}')
    return out

t0 = time.time()
use = np.concatenate([tr, va, te])
use_seqs = list(seq_arr[use])
if EMBED_BACKEND == 'kmer':
    emb_used = kmer_embed(use_seqs)
elif EMBED_BACKEND == 'esm':
    emb_used = esm_embed(use_seqs)
elif EMBED_BACKEND == 'precomputed':
    full = np.load(PROC / SEQ_EMB_FILE, mmap_mode='r')
    emb_used = np.asarray(full[use])
else:
    raise ValueError(EMBED_BACKEND)
# map back to a dict keyed by global row index for the dataset
emb = {int(g): emb_used[i] for i, g in enumerate(use)}
EMB_DIM = emb_used.shape[1]
print(f'embedding backend={EMBED_BACKEND}  dim={EMB_DIM}  ({time.time()-t0:.1f}s)')

embedding backend=precomputed  dim=480  (30.2s)


## Sequence-only ranker

In [5]:
# standardize embeddings on train
tr_mat = np.stack([emb[int(g)] for g in tr])
mu = tr_mat.mean(0); sd = tr_mat.std(0) + 1e-6

class EmbDataset(Dataset):
    def __init__(self, ix): self.ix = ix
    def __len__(self): return len(self.ix)
    def __getitem__(self, i):
        g = int(self.ix[i])
        x = (emb[g] - mu) / sd
        return torch.from_numpy(x.astype(np.float32)), torch.tensor(y_all[g], dtype=torch.float32)

BATCH = 2048
tr_dl = DataLoader(EmbDataset(tr), batch_size=BATCH, shuffle=True)
va_dl = DataLoader(EmbDataset(va), batch_size=BATCH)
te_dl = DataLoader(EmbDataset(te), batch_size=BATCH)

pos_weight = torch.tensor([(len(tr) - y_all[tr].sum()) / max(y_all[tr].sum(), 1)],
                          dtype=torch.float32, device=device)

class SeqMLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x): return self.net(x)

def predict(model, dl):
    model.eval(); out = []
    with torch.no_grad():
        for xb, _ in dl:
            out.append(torch.sigmoid(model(xb.to(device))).squeeze(-1).cpu().numpy())
    return np.concatenate(out)

def train(model, epochs=10, lr=1e-3, wd=1e-5):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best, best_state = -1, None
    for ep in range(1, epochs + 1):
        model.train(); ls = 0; n = 0
        for xb, yb in tr_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = loss_fn(model(xb).squeeze(-1), yb)
            loss.backward(); opt.step(); ls += loss.item()*len(xb); n += len(xb)
        va_ap = average_precision_score(y_all[va], predict(model, va_dl))
        print(f'  ep {ep}/{epochs}  loss={ls/n:.4f}  val AUPRC={va_ap:.4f}')
        if va_ap > best: best, best_state = va_ap, {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); return model

print(f'training sequence-only ranker (backend={EMBED_BACKEND}, dim={EMB_DIM}) ...')
seq_model = train(SeqMLP(EMB_DIM), epochs=10)
seq_score_te = predict(seq_model, te_dl)
seq_score_va = predict(seq_model, va_dl)

training sequence-only ranker (backend=precomputed, dim=480) ...


  ep 1/10  loss=0.8422  val AUPRC=0.2016


  ep 2/10  loss=0.7982  val AUPRC=0.2145


  ep 3/10  loss=0.7777  val AUPRC=0.2197


  ep 4/10  loss=0.7646  val AUPRC=0.2265


...

  ep 6/10  loss=0.7428  val AUPRC=0.2293


  ep 7/10  loss=0.7322  val AUPRC=0.2355


  ep 8/10  loss=0.7218  val AUPRC=0.2418


  ep 9/10  loss=0.7152  val AUPRC=0.2325


  ep 10/10  loss=0.7066  val AUPRC=0.2399


## Abundance-only (v0) scores + fusion

We reload the v0 abundance MLP (`predictions/02_mlp.pt`) and score the val + test families, recomputing the same `log1p` + train-standardization. The fusion model is a logistic regression on `[abundance_score, sequence_score]`, fit on **val** and evaluated on **test** — so no test leakage. In SMOKE mode, if the abundance matrix isn't loadable for these indices we fall back to the saved v0 test predictions for the abundance score and skip the val-fitted fusion (report rank-average instead).

In [6]:
def score_abundance(model_path, indices):
    """Reload v0 MLP and score given family indices from the mmap abundance matrix."""
    X = np.load(PROC / 'abundance.npy', mmap_mode='r')
    n_feat = X.shape[1]
    # recompute train standardization stats (identical to 02)
    fsum = np.zeros(n_feat); fsq = np.zeros(n_feat); B = 16_384
    tr_sorted = np.sort(tr_full)
    for s in range(0, len(tr_sorted), B):
        rows = np.log1p(X[tr_sorted[s:s+B]].astype(np.float32))
        fsum += rows.sum(0); fsq += (rows**2).sum(0)
    fmean = (fsum/len(tr_full)).astype(np.float32)
    fstd = np.sqrt(np.maximum(fsq/len(tr_full) - fmean**2, 1e-12)).astype(np.float32)
    class MLP(nn.Module):
        def __init__(s, d):
            super().__init__()
            s.net = nn.Sequential(nn.Linear(d,256), nn.ReLU(), nn.Dropout(0.3),
                                  nn.Linear(256,64), nn.ReLU(), nn.Linear(64,1))
        def forward(s,x): return s.net(x)
    m = MLP(n_feat).to(device); m.load_state_dict(torch.load(model_path, map_location=device)); m.eval()
    out = np.empty(len(indices), dtype=np.float32)
    with torch.no_grad():
        for s in range(0, len(indices), 4096):
            chunk = indices[s:s+4096]
            xb = np.log1p(X[np.sort(chunk)].astype(np.float32))
            # note: np.sort reorders; map back
            order = np.argsort(chunk); inv = np.argsort(order)
            xb = (xb - fmean) / fstd
            sc = torch.sigmoid(m(torch.from_numpy(xb).to(device))).squeeze(-1).cpu().numpy()
            out[s:s+len(chunk)] = sc[inv]
    return out

# full (non-subsampled) train indices needed for correct standardization stats
_tr_full, _te_full = train_test_split(np.arange(len(y_all)), test_size=0.15, stratify=y_all, random_state=0)
tr_full, _ = train_test_split(_tr_full, test_size=0.15/0.85, stratify=y_all[_tr_full], random_state=0)

abund_ok = (PROC / '02_mlp.pt').exists() or (PRED / '02_mlp.pt').exists()
mlp_path = (PRED / '02_mlp.pt') if (PRED / '02_mlp.pt').exists() else (PROC / '02_mlp.pt')
fusion_mode = None
try:
    if not abund_ok: raise FileNotFoundError('02_mlp.pt not found')
    abund_te = score_abundance(mlp_path, te)
    abund_va = score_abundance(mlp_path, va)
    from sklearn.linear_model import LogisticRegression
    Xva = np.c_[abund_va, seq_score_va]; Xte = np.c_[abund_te, seq_score_te]
    fuse = LogisticRegression(max_iter=1000).fit(Xva, y_all[va])
    fusion_score_te = fuse.predict_proba(Xte)[:, 1]
    fusion_mode = 'val-fitted logreg'
except Exception as e:
    print(f'  [fusion fallback] {type(e).__name__}: {e}')
    if v0 is not None:
        m = dict(zip(v0['family_id'], v0['mlp_score']))
        abund_te = np.array([m.get(f, np.nan) for f in fids[te]], dtype=float)
    else:
        abund_te = np.full(len(te), np.nan)
    def rank01(a):
        a = pd.Series(a); return a.rank(pct=True).values
    fusion_score_te = np.nanmean(np.c_[rank01(abund_te), rank01(seq_score_te)], axis=1)
    fusion_mode = 'rank-average (fallback)'
print(f'fusion mode: {fusion_mode}')

fusion mode: val-fitted logreg


## Results

In [7]:
y_te = y_all[te]
def report(name, sc):
    sc = np.asarray(sc, dtype=float)
    ok = ~np.isnan(sc)
    out = {'method': name, 'auprc': average_precision_score(y_te[ok], sc[ok]),
           'auroc': roc_auc_score(y_te[ok], sc[ok])}
    for k in (100, 1000):
        k = min(k, ok.sum()-1)
        top = np.argpartition(-sc[ok], k)[:k]
        out[f'P@{k}'] = y_te[ok][top].mean(); out[f'enr@{k}'] = out[f'P@{k}']/max(y_te[ok].mean(),1e-9)
    return out

rows = [report(f'Sequence-only ({EMBED_BACKEND})', seq_score_te)]
rows.append(report('MetaWIBELE unsup', labels['metawibele_unsup_rank'].fillna(0).values[te]))
rows.append(report('MetaWIBELE sup', labels['metawibele_sup_rank'].fillna(0).values[te]))
if not np.isnan(abund_te).all():
    rows.append(report('Abundance v0 (MLP)', abund_te))
rows.append(report(f'Fusion [{fusion_mode}]', fusion_score_te))
res = pd.DataFrame(rows).set_index('method').round(4)
print(res.to_string())
print(f'\nbest AUPRC: {res["auprc"].idxmax()} ({res["auprc"].max():.4f})')

                              auprc   auroc  P@100  enr@100  P@1000  enr@1000
method                                                                       
Sequence-only (precomputed)  0.2410  0.9021   0.61  32.6727   0.452   24.2099
MetaWIBELE unsup             0.0755  0.7296   0.35  18.7466   0.212   11.3551
MetaWIBELE sup               0.0640  0.7029   0.32  17.1398   0.198   10.6052
Abundance v0 (MLP)           0.1424  0.8315   0.41  21.9603   0.339   18.1574
Fusion [val-fitted logreg]   0.3586  0.9248   0.91  48.7412   0.641   34.3331

best AUPRC: Fusion [val-fitted logreg] (0.3586)


In [8]:
tag = 'smoke' if SMOKE else EMBED_BACKEND
res.to_csv(PRED / f'03_results_{tag}.csv')
pd.DataFrame({'family_id': fids[te], 'y_true': y_te,
              'seq_score': seq_score_te, 'fusion_score': fusion_score_te}).to_parquet(
    PRED / f'03_predictions_{tag}.parquet', index=False)
print(f'wrote predictions/03_results_{tag}.csv and 03_predictions_{tag}.parquet')
if SMOKE:
    assert res.loc[f'Sequence-only ({EMBED_BACKEND})', 'auprc'] > y_te.mean(), 'sequence model no better than random'
    print('\nSMOKE TEST PASSED — pipeline runs end-to-end; sequence model beats random base rate.')

wrote predictions/03_results_precomputed.csv and 03_predictions_precomputed.parquet


## Running for real on CSUC (ESM-2 650M)

The k-mer backend above verifies the pipeline. For the real result, compute ESM-2 embeddings on the cluster GPU **once**, then rerun this notebook with `SMOKE=False, EMBED_BACKEND='precomputed'`.

**Step 1 — copy data + script to the cluster:**
```bash
scp -P 2122 processed/sequences.parquet uvicommsc29@pirineus3.csuc.cat:~/taskb/processed/
scp -P 2122 esm_embed.py run_esm_cluster.sh uvicommsc29@pirineus3.csuc.cat:~/taskb/
```

**Step 2 — submit the embedding job** (`run_esm_cluster.sh` is a SLURM template in this folder). It writes `processed/seq_emb.npy` (1.45 M × 1280 float32 ≈ 7.4 GB for the 650M model).

**Step 3 — bring embeddings back and rerun locally (or train on cluster):**
```bash
scp -P 2122 uvicommsc29@pirineus3.csuc.cat:~/taskb/processed/seq_emb.npy processed/
```
then set `SMOKE=False; EMBED_BACKEND='precomputed'` in the config cell and run all.

**Note on memory:** 7.4 GB of embeddings won't fit the laptop's remaining disk right now; plan to train the sequence + fusion models *on the cluster* too (this notebook runs there unchanged), and only copy back the small `03_results_*.csv` / `03_predictions_*.parquet`.